In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "enviroment_bj").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("project_root:", ROOT)

project_root: C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl


In [2]:
from copy import deepcopy


from enviroment_bj import BlackjackEnvironment, BlackjackConfig, ObservationConfig, StartStateConfig
from loss import BellmanLossConfig, LossPhaseWeightConfig
from model.agents import DuelingRecurrentDoubleDQN, FeedForwardDoubleDQN , RecurrentDoubleDQN
from training import (
    train_model,
    ReplayBufferConfig,
    EpsilonScheduleConfig,
    DualEpsilonConfig,
    NStepConfig,
    OptimizationConfig,
    TargetUpdateConfig,
    EvaluationConfig,
    CheckpointConfig,
    PrintConfig,
    TrainerConfig,
    TrainingPipelineConfig,
)
from model.encoder import BlackjackObservationEncoder, EncoderConfig
from model.agents import DuelingRecurrentDoubleDQN, AgentNetworkConfig


In [8]:
# ============================================================
# OBSERVATION
# ------------------------------------------------------------
# Volvemos a observación realista para shoe desconocido.
# Pero todavía sin recent_actions y sin exact shoe composition.
# ============================================================

observation_config = ObservationConfig(
    profile="table_realistic_unknown_progress",
    obs_include_table_rules=True,
    obs_include_visible_rules_only=True,
    obs_include_hidden_rules=False,
    obs_include_decision_phase=True,
    obs_include_available_bet_multipliers=True,
    obs_current_hand_mode="table_raw",
    obs_include_other_player_hands=True,
    obs_include_current_bet=True,
    obs_include_betting_context=True,
    obs_include_hand_context=True,
    obs_include_insurance_context=True,
    obs_include_temporal_context=True,
    obs_include_hands_since_shuffle=False,
    obs_include_estimated_shoe_progress=False,
    obs_include_last_hand_outcome=False,
    obs_include_recent_actions=False,
    obs_recent_actions_window=8,
    obs_include_observed_cards_history=True,
    obs_observed_cards_mode="rank_counts",
    obs_recent_cards_window=32,
    obs_reset_history_on_shuffle=True,
    obs_include_exact_shoe_composition=False,
    obs_include_discard_summary=True,
    obs_include_n_decks=False,
    obs_include_shoe_penetration_rule=False,
)

# ============================================================
# START STATE
# ------------------------------------------------------------
#  mesa empezada.
# ============================================================

start_state_config = StartStateConfig(
    mode="fresh_shoe",
    min_burned_rounds=0,
    max_burned_rounds=0,
    clear_visible_histories_after_burn=True,
    hide_reshuffle_progress_from_observation=False,
)

# ============================================================
# TABLE CONFIG
# ------------------------------------------------------------
# Seguimos realistas, pero:
# - todavía solo 1x
# - sin exogenous cards
# - sin insurance por ahora
# - sin six-card charlie por ahora
# ============================================================

blackjack_config = BlackjackConfig(
    n_decks=8,
    shoe_penetration=0.75,
    use_cut_card=True,
    visible_shoe_change=True,

    exogenous_cards=False,
    simulate_exogenous_visible_cards=False,
    exogenous_visible_cards_mode="disabled",

    dealer_hits_soft_17=False,          # S17
    blackjack_payout=1.5,               # 3:2
    dealer_peeks_for_blackjack=True,

    double_allowed_on="any_two_cards",
    double_after_split_allowed=True,
    double_split_aces_allowed=False,

    split_rule="same_value",
    max_hands_after_split=2,
    max_split_depth_per_hand=1,
    resplit_aces_allowed=False,
    hit_split_aces_allowed=False,

    surrender_allowed=False,
    insurance_allowed=False,
    six_card_charlie_enabled=False,

    base_bet=1.0,
    bet_multipliers=(1,),
    strict_shoe_validation=False,

    observation=observation_config,
    observation_mode=None,
    expose_shoe_composition=False,
)

# ============================================================
# ENVS
# ------------------------------------------------------------
# Dos penetraciones para no meter demasiada variabilidad,
# pero tampoco dejarlo totalmente fijo.
# ============================================================

base_seed = 1
penetrations = [0.70, 0.80]

envs = []
for i, penetration in enumerate(penetrations):
    cfg = deepcopy(blackjack_config)
    cfg.shoe_penetration = penetration

    env = BlackjackEnvironment(
        config=cfg,
        seed=base_seed + i,
        start_state=start_state_config,
    )
    envs.append(env)


In [ ]:
# ============================================================
# ENCODER
# ------------------------------------------------------------
# Fase puente:
# - más realista que el baseline
# - unknown_progress
# - memoria del shoe observada
# - todavía sin acciones recientes ni exact shoe
# ============================================================

encoder_config = EncoderConfig(
    profile="table_realistic_unknown_progress",

    encode_rules=True,
    encode_betting_context=True,
    encode_other_hands=True,

    # apagados temporalmente
    encode_temporal=False,
    encode_observed_history=False,
    encode_discard_summary=False,
    encode_recent_actions=False,

    encode_exact_shoe=False,
    encode_action_mask_features=False,

    card_encoding="one_hot_rank",
    history_encoding="rank_counts",
    normalize_counts=True,
    use_visible_table_rules_only=True,

    max_current_hand_cards=12,
    max_cards_per_hand=12,
    max_other_hands=4,
    max_recent_actions=5,
    max_recent_cards=32,
    max_recent_discard_cards=16,
)


encoder = BlackjackObservationEncoder(config=encoder_config)

# ============================================================
# MODEL
# ------------------------------------------------------------
# Recurrente para el régimen parcialmente observable.
# GRU contenida para no disparar inestabilidad.
# ============================================================

model_config = AgentNetworkConfig.for_architecture(
    architecture="feedforward",
    encoder_profile=encoder_config.profile,
    activation="relu",
    use_layer_norm=False,
    dropout=0.0,
    feedforward_hidden_dims=(256, 256, 128),
    use_phase_adapters=False,
    use_module_gating=False,
)

model = FeedForwardDoubleDQN(
    config=model_config,
    encoder=encoder,
)

# ============================================================
# EPSILON
# ------------------------------------------------------------
# Betting casi irrelevante porque solo existe 1x.
# Playing con algo más de exploración que en tu corrida final
# del baseline para ayudar a double/split en el régimen realista.
# ============================================================

dual_epsilon_config = DualEpsilonConfig(
    betting=EpsilonScheduleConfig(
        start=0.15,
        end=0.02,
        decay_steps=30_000,
        evaluation_epsilon=0.0,
    ),
    playing=EpsilonScheduleConfig(
        start=0.18,
        end=0.05,
        decay_steps=90_000,
        evaluation_epsilon=0.0,
    ),
)

# ============================================================
# LOSS
# ------------------------------------------------------------
# Seguimos priorizando playing.
# ============================================================

loss_config = BellmanLossConfig(
    gamma=0.99,
    loss_type="huber",
    validate_current_actions=True,
    validate_next_action_mask=True,
    allow_terminal_without_legal_next_action=True,
    phase_weights=LossPhaseWeightConfig(
        enabled=True,
        betting_weight=0.25,
        playing_weight=1.50,
    ),
)

# ============================================================
# N-STEP
# ============================================================

n_step_config = NStepConfig(
    enabled=True,
    n_steps=3,
)

# ============================================================
# REPLAY BUFFER
# ------------------------------------------------------------
# Ahora sí importa la secuencia.
# ============================================================

replay_buffer_config = ReplayBufferConfig(
    capacity=150_000,
    batch_size=64,
    warmup_size=12_000,
    sequence_length=16,
    min_sequence_length=8,
)

# ============================================================
# OPTIMIZATION
# ============================================================

optimization_config = OptimizationConfig(
    optimizer="adamw",
    learning_rate=1e-4,
    weight_decay=1e-5,
    scheduler="step",
    scheduler_step_size=25_000,
    scheduler_gamma=0.97,
    gradient_clipping=True,
    max_grad_norm=5.0,
)

# ============================================================
# TARGET UPDATE
# ------------------------------------------------------------
# Hard update más espaciado para más estabilidad.
# ============================================================

target_update_config = TargetUpdateConfig(
    mode="hard",
    hard_update_interval=1000,
    soft_tau=0.005,
)

# ============================================================
# EVALUATION
# ============================================================

evaluation_config = EvaluationConfig(
    enabled=True,
    every_n_epochs=1,
    num_rounds=2500,
    max_decisions=25_000,
)

# ============================================================
# CHECKPOINTS
# ============================================================

checkpoint_config = CheckpointConfig(
    directory=r"training_checkpoints\bridge_unknown_progress_recurrent_v1",
    save_latest=True,
    save_best_eval=True,
    save_periodic=True,
    periodic_interval_updates=2500,
    best_metric_name="ev_per_1000_hands",
    maximize_best_metric=True,
)

# ============================================================
# PRINTS
# ============================================================

print_config = PrintConfig(
    enable=True,
    print_run_summary=True,
    print_warmup_interval=1000,
    print_update_interval=200,
    print_collection_interval=1000,
    print_epoch_header=True,
    print_epoch_summary=True,
    print_eval_summary=True,
    include_segment_details=False,
)

# ============================================================
# TRAINER
# ------------------------------------------------------------
# Aquí sí dejamos secuencias terminar en done para que la GRU
# no mezcle demasiado entre rounds al inicio de esta fase puente.
# ============================================================

trainer_config = TrainerConfig(
    total_epochs=25,
    env_steps_per_epoch=4500,
    train_frequency=4,
    updates_per_train_step=1,
    max_updates_per_epoch=None,
    device="cpu",
    seed=44,
    reset_hidden_on_round_end=False,
    sequence_end_on_done=False,
    flush_partial_sequences_at_epoch_end=True,
    loss=loss_config,
)
# ============================================================
# PIPELINE
# ============================================================

pipeline_config = TrainingPipelineConfig(
    trainer=trainer_config,
    replay_buffer=replay_buffer_config,
    epsilon=dual_epsilon_config,
    n_step=n_step_config,
    optimization=optimization_config,
    target_update=target_update_config,
    evaluation=evaluation_config,
    checkpoints=checkpoint_config,
    prints=print_config,
)


In [ ]:
result = train_model(
    envs=envs,
    model=model,
    pipeline_config=pipeline_config,
    resume=False,
    resume_checkpoint_path=None,
)

BLACKJACK RL RUN
  Model      : arch=feedforward | recurrent=none | encoder=table_realistic_unknown_progress | obs=table_realistic_unknown_progress | start=fresh_shoe
  Runtime    : device=cpu | epochs=25 | envs=2 | steps/epoch=9000 | updates/epoch~=2250 | params=440,980
  Optim      : optimizer=adamw | lr=1.00e-04 | loss=huber | gamma=0.9900 | grad_clip=True(5.00)
  Replay     : warmup=12000 | capacity=150000 | batch=64 | seq_len=16 | min_seq_len=8
  Explore    : eps_bet=0.150->0.020 (decay 30000) | eps_play=0.180->0.050 (decay 90000) | target=hard | interval=1000 | tau=0.0050
  Extras     : n_step=True(3) | phase_loss_w=True (bet 0.25, play 1.50) | phase_adapters=True | module_gating=False
  Eval / CKPT: eval_rounds=2500 | eval_decisions=25000 | checkpoints=training_checkpoints\bridge_unknown_progress_recurrent_v1
  Table      : decks=8 | pen=0.70 | S17=True | payout=1.50 | double=any_two_cards | split=same_value | DAS=True

=== Epoch 1/25 ===
[Warmup] buffer 1000/12000
[Warmup] buff

KeyboardInterrupt: 